# Lab 5 — Can a Transformer Generate Good Quality Code?

**Programming Language:** Java

## Research Question

How does the choice of Transformer model affect the quality of Java code generated for programming tasks of different difficulty levels?

## Hypothesis

I expect code-specialized and instruction-tuned Transformer models to generate more correct and robust Java code than a general-purpose Transformer model, especially as task difficulty increases.

#### Installer les librairies

In [1]:
!pip -q install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 42.4 MB/s eta 0:00:00


#### Définir les modèles

> Ajouter une citation



In [2]:
MODELS = {
    "DistilGPT2": "distilbert/distilgpt2",
    "CodeGen-Multi 350M": "Salesforce/codegen-350M-multi",
    "Qwen2.5-Coder 0.5B Instruct":
        "Qwen/Qwen2.5-Coder-0.5B-Instruct"
}

MODELS

{'DistilGPT2': 'distilbert/distilgpt2',
 'CodeGen-Multi 350M': 'Salesforce/codegen-350M-multi',
 'Qwen2.5-Coder 0.5B Instruct': 'Qwen/Qwen2.5-Coder-0.5B-Instruct'}

## Task 1 — Even or Odd Number

In [6]:
TASK1_PROMPT = """
Generate valid Java source code.

Task:
Create a program that determines whether an integer is even or odd.

Requirements:
- Use a class named Main.
- Read one integer from standard input.
- Print exactly "Even" if the number is even.
- Print exactly "Odd" if the number is odd.
- Do not print explanations or additional text.
- Return source code only.
"""

TASK1_TESTS = [
    ("4", "Even"),
    ("7", "Odd"),
    ("0", "Even"),
    ("-3", "Odd")
]

print(TASK1_PROMPT)


Generate valid Java source code.

Task:
Create a program that determines whether an integer is even or odd.

Requirements:
- Use a class named Main.
- Read one integer from standard input.
- Print exactly "Even" if the number is even.
- Print exactly "Odd" if the number is odd.
- Do not print explanations or additional text.
- Return source code only.



#### Code Generation with the Three Transformers

In [7]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


def generate_code(model_id, prompt):
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        trust_remote_code=True
    ).to(DEVICE)

    # Use the chat format only when the model provides one
    if tokenizer.chat_template:
        formatted_prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True
        )
    else:
        formatted_prompt = prompt

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=500,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    generated_code = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    # Free memory before loading the next model
    del model
    del tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return generated_code


TASK1_OUTPUTS = {}

for model_name, model_id in MODELS.items():

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    code = generate_code(model_id, TASK1_PROMPT)

    TASK1_OUTPUTS[model_name] = code

    print(code)

Device: cpu

DistilGPT2


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
-

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-multi
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Output:
- The program should print "Even" if the number is even.
- The program should print "Odd" if the number is odd.
- The program should print "Even" if the number is even.
- The program should print "Odd" if the number is odd.
- The program should print "Even" if the number is even.
- The program should print "Odd" if the number is odd.
- The program should print "Even" if the number is even.
- The program should print "Odd" if the number is odd.
- The program should print "Even" if the number is even.
- The program should print "Odd" if the number is odd.
- The program should print "Even" if the number is even.
- The program should print "Odd" if the number is odd.
- The program should print "Even" if the number is even.
- The program should print "Odd" if the number is odd.
- The program should print "Even" if the number is even.
- The program should print "Odd" if the number is odd.
- The program should print "Even" if the number is even.
- The program should print "Odd" if the

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

```java
import java.util.Scanner;

public class EvenOddChecker {
    public static void main(String[] args) {
        Scanner scanner = new Scanner(System.in);
        System.out.print("Enter an integer: ");
        int number = scanner.nextInt();
        
        if (number % 2 == 0) {
            System.out.println("Even");
        } else {
            System.out.println("Odd");
        }
        
        scanner.close();
    }
}
```

This Java program reads an integer from the user, checks if it is even or odd, and prints the appropriate message. The `Scanner` class is used to read input from the user.


## Task 2 — Maximum Value in an Array

In [9]:
TASK2_PROMPT = """
Generate valid Java source code.

Task:
Create a program that finds the maximum value in an integer array.

Requirements:
- Use a class named Main.
- Read an integer n representing the array size.
- Read n integers.
- Print only the maximum value.
- Do not print explanations or additional text.
- Return source code only.
"""

TASK2_TESTS = [
    ("5\n3 8 2 10 4", "10"),
    ("4\n-5 -2 -9 -1", "-1"),
    ("1\n7", "7"),
    ("5\n4 4 4 4 4", "4")
]

print(TASK2_PROMPT)


Generate valid Java source code.

Task:
Create a program that finds the maximum value in an integer array.

Requirements:
- Use a class named Main.
- Read an integer n representing the array size.
- Read n integers.
- Print only the maximum value.
- Do not print explanations or additional text.
- Return source code only.



##  Code Generation with the Three Transformers

In [10]:
TASK2_OUTPUTS = {}

for model_name, model_id in MODELS.items():

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    code = generate_code(model_id, TASK2_PROMPT)

    TASK2_OUTPUTS[model_name] = code

    print(code)


DistilGPT2


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
-

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-multi
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Input Format:

The first line contains an integer n representing the array size.

The second line contains an integer n representing the integer array.

Output Format:

The first line contains an integer n representing the maximum value.

The second line contains an integer n representing the integer array.

Sample Input:

1
2
3

Sample Output:

3

Explanation:

The maximum value is 3.

Explanation:

The maximum value is 3.

Explanation:

The maximum value is 3.

Explanation:

The maximum value is 3.

Explanation:

The maximum value is 3.

Explanation:

The maximum value is 3.

Explanation:

The maximum value is 3.

Explanation:

The maximum value is 3.

Explanation:

The maximum value is 3.

Explanation:

The maximum value is 3.

Explanation:

The maximum value is 3.

Explanation:

The maximum value is 3.

Explanation:

The maximum value is 3.

Explanation:

The maximum value is 3.

Explanation:

The maximum value is 3.

Explanation:

The maximum value is 3.

Explanation:

The maximum

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

```java
public class MaxValueFinder {
    public static void main(String[] args) {
        // Read the size of the array
        int n = Integer.parseInt(args[0]);
        
        // Read the array elements
        int[] numbers = new int[n];
        for (int i = 0; i < n; i++) {
            numbers[i] = Integer.parseInt(args[i + 1]);
        }
        
        // Find and print the maximum value
        int max = numbers[0];
        for (int num : numbers) {
            if (num > max) {
                max = num;
            }
        }
        
        // Print the maximum value
        System.out.println("The maximum value is: " + max);
    }
}
```

This Java program defines a class `MaxValueFinder` with a static method `main` that takes two arguments: the size of the array and the array itself. It reads these values from the command line arguments, finds the maximum value in the array, and prints it. The program does not include any explanations or additional text, as it simply pr

## Task 3 — Second Largest Distinct Number

In [11]:
TASK3_PROMPT = """
Generate valid Java source code.

Task:
Create a program that finds the second largest distinct value in an integer array.

Requirements:
- Use a class named Main.
- Read an integer n representing the array size.
- Read n integers from standard input.
- Print only the second largest distinct value.
- If there is no second distinct value, print "No second value".
- Do not print explanations or additional text.
- Return source code only.
"""

TASK3_TESTS = [
    ("5\n3 8 2 10 4", "8"),
    ("6\n5 5 4 4 3 3", "4"),
    ("4\n7 7 7 7", "No second value"),
    ("5\n-10 -3 -7 -3 -5", "-5")
]

print(TASK3_PROMPT)


Generate valid Java source code.

Task:
Create a program that finds the second largest distinct value in an integer array.

Requirements:
- Use a class named Main.
- Read an integer n representing the array size.
- Read n integers from standard input.
- Print only the second largest distinct value.
- If there is no second distinct value, print "No second value".
- Do not print explanations or additional text.
- Return source code only.



#### Code Generation with the Three Transformers

In [12]:
TASK3_OUTPUTS = {}

for model_name, model_id in MODELS.items():

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    code = generate_code(model_id, TASK3_PROMPT)

    TASK3_OUTPUTS[model_name] = code

    print(code)


DistilGPT2


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
- Return source code only.
-

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-multi
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Output:
The second largest distinct value is printed.

Sample Input:

1
2
3

Sample Output:

2

"""

class Solution(object):
    def findLargest(self, nums):
        """
        :type nums: List[int]
        :rtype: int
        """
        n = len(nums)
        if n == 0:
            return 0
        if n == 1:
            return nums[0]
        if n == 2:
            return nums[1]
        if n == 3:
            return nums[2]
        if n == 4:
            return nums[3]
        if n == 5:
            return nums[4]
        if n == 6:
            return nums[5]
        if n == 7:
            return nums[6]
        if n == 8:
            return nums[7]
        if n == 9:
            return nums[8]
        if n == 10:
            return nums[9]
        if n == 11:
            return nums[10]
        if n == 12:
            return nums[11]
        if n == 13:
            return nums[12]
        if n == 14:
            return nums[13]
        if n == 15:
            return nums[14]
     

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Certainly! Below is a Java program that accomplishes the task of finding the second largest distinct value in an integer array. The program reads an integer `n` and then reads `n` integers from standard input. It prints the second largest distinct value, or "No second value" if there is no such value.

```java
import java.util.*;

public class SecondLargestDistinctValue {
    public static void main(String[] args) {
        Scanner scanner = new Scanner(System.in);
        
        // Read the size of the array
        System.out.print("Enter the size of the array: ");
        int n = scanner.nextInt();
        
        // Create a HashSet to store unique elements
        Set<Integer> uniqueElements = new HashSet<>();
        
        // Read the elements into the set
        for (int i = 0; i < n; i++) {
            System.out.print("Enter element " + (i + 1) + ": ");
            int element = scanner.nextInt();
            uniqueElements.add(element);
        }
        
        // Ch

## Evaluation — Execute Predefined Tests

In [14]:
import re
import subprocess
import tempfile
import os

def extract_code(text):
    # If the model returned ```java ... ```
    match = re.search(r"```(?:java)?\s*(.*?)```", text, re.DOTALL)
    if match:
        return match.group(1).strip()

    # Otherwise keep the raw generated output
    return text.strip()


def evaluate_java(code, tests):
    code = extract_code(code)

    # Detect public class name
    match = re.search(r"public\s+class\s+(\w+)", code)

    if not match:
        return {
            "Compilation": "Fail",
            "Tests": [],
            "Reason": "No valid public Java class found"
        }

    class_name = match.group(1)

    with tempfile.TemporaryDirectory() as tmpdir:

        java_file = os.path.join(tmpdir, f"{class_name}.java")

        with open(java_file, "w") as f:
            f.write(code)

        # Compile
        compile_result = subprocess.run(
            ["javac", java_file],
            capture_output=True,
            text=True
        )

        if compile_result.returncode != 0:
            return {
                "Compilation": "Fail",
                "Tests": [],
                "Reason": compile_result.stderr
            }

        results = []

        for input_data, expected in tests:

            run_result = subprocess.run(
                ["java", "-cp", tmpdir, class_name],
                input=input_data + "\n",
                capture_output=True,
                text=True,
                timeout=5
            )

            observed = run_result.stdout.strip()

            results.append({
                "Input": input_data,
                "Expected": expected,
                "Observed": observed,
                "Result": "Pass" if observed == expected else "Fail"
            })

        return {
            "Compilation": "Pass",
            "Tests": results
        }

In [16]:
import re
import os
import subprocess
import tempfile


# --------------------------------------------------
# 1. Extract Java code from the model response
# --------------------------------------------------

def extract_code(text):
    # Extract code inside ```java ... ```
    match = re.search(r"```(?:java)?\s*(.*?)```", text, re.DOTALL)

    if match:
        return match.group(1).strip()

    # Otherwise keep the complete generated text
    return text.strip()


# --------------------------------------------------
# 2. Remove Java / Colab environment warnings
# --------------------------------------------------

def clean_output(text):
    clean_lines = []

    for line in text.splitlines():
        if "[warning][os,container]" not in line:
            clean_lines.append(line)

    return "\n".join(clean_lines).strip()


# --------------------------------------------------
# 3. Compile and test one generated Java program
# --------------------------------------------------

def evaluate_java(code, tests):

    code = extract_code(code)

    # Find a public Java class
    class_match = re.search(
        r"public\s+class\s+([A-Za-z_][A-Za-z0-9_]*)",
        code
    )

    if not class_match:
        return {
            "Compilation": "Fail",
            "Runtime": "Not tested",
            "Tests": [],
            "Reason": "No valid public Java class found."
        }

    class_name = class_match.group(1)

    with tempfile.TemporaryDirectory() as tmpdir:

        java_file = os.path.join(
            tmpdir,
            f"{class_name}.java"
        )

        # Save generated code
        with open(java_file, "w", encoding="utf-8") as f:
            f.write(code)

        # -----------------------------
        # Compile
        # -----------------------------

        compile_result = subprocess.run(
            ["javac", java_file],
            capture_output=True,
            text=True
        )

        if compile_result.returncode != 0:

            error = clean_output(
                compile_result.stderr + "\n" +
                compile_result.stdout
            )

            return {
                "Compilation": "Fail",
                "Runtime": "Not tested",
                "Tests": [],
                "Reason": error if error else "Compilation failed."
            }

        # -----------------------------
        # Execute predefined tests
        # -----------------------------

        test_results = []

        for input_data, expected in tests:

            try:

                run_result = subprocess.run(
                    ["java", "-cp", tmpdir, class_name],
                    input=input_data + "\n",
                    capture_output=True,
                    text=True,
                    timeout=5
                )

                observed = clean_output(run_result.stdout)
                runtime_error = clean_output(run_result.stderr)

                # Program crashed
                if run_result.returncode != 0:

                    test_results.append({
                        "Input": input_data,
                        "Expected": expected,
                        "Observed": observed,
                        "Result": "Fail",
                        "Reason": (
                            runtime_error
                            if runtime_error
                            else "Runtime error"
                        )
                    })

                # Program executed normally
                else:

                    test_results.append({
                        "Input": input_data,
                        "Expected": expected,
                        "Observed": observed,
                        "Result":
                            "Pass"
                            if observed == expected
                            else "Fail",
                        "Reason": ""
                    })

            except subprocess.TimeoutExpired:

                test_results.append({
                    "Input": input_data,
                    "Expected": expected,
                    "Observed": "",
                    "Result": "Fail",
                    "Reason": "Execution timeout"
                })

        return {
            "Compilation": "Pass",
            "Runtime": "Tested",
            "Tests": test_results,
            "Reason": ""
        }


# --------------------------------------------------
# 4. Group the three tasks
# --------------------------------------------------

ALL_OUTPUTS = {
    "Task 1": (TASK1_OUTPUTS, TASK1_TESTS),
    "Task 2": (TASK2_OUTPUTS, TASK2_TESTS),
    "Task 3": (TASK3_OUTPUTS, TASK3_TESTS)
}


# --------------------------------------------------
# 5. Evaluate all models on all tasks
# --------------------------------------------------

ALL_RESULTS = {}

for task_name, (outputs, tests) in ALL_OUTPUTS.items():

    print("\n" + "#" * 80)
    print(task_name)
    print("#" * 80)

    ALL_RESULTS[task_name] = {}

    for model_name, code in outputs.items():

        print("\n" + "-" * 70)
        print(model_name)
        print("-" * 70)

        result = evaluate_java(code, tests)

        ALL_RESULTS[task_name][model_name] = result

        print("Compilation:", result["Compilation"])

        # Compilation failed
        if result["Compilation"] == "Fail":
            print("Reason:", result["Reason"][:500])
            continue

        # Compilation successful
        for i, test in enumerate(result["Tests"], start=1):

            print(f"\nTest {i}")
            print("Input:", repr(test["Input"]))
            print("Expected:", repr(test["Expected"]))
            print("Observed:", repr(test["Observed"]))
            print("Result:", test["Result"])

            if test["Reason"]:
                print("Reason:", test["Reason"][:500])


################################################################################
Task 1
################################################################################

----------------------------------------------------------------------
DistilGPT2
----------------------------------------------------------------------
Compilation: Fail
Reason: No valid public Java class found.

----------------------------------------------------------------------
CodeGen-Multi 350M
----------------------------------------------------------------------
Compilation: Fail
Reason: No valid public Java class found.

----------------------------------------------------------------------
Qwen2.5-Coder 0.5B Instruct
----------------------------------------------------------------------
Compilation: Pass

Test 1
Input: '4'
Expected: 'Even'
Observed: 'Enter an integer: Even'
Result: Fail

Test 2
Input: '7'
Expected: 'Odd'
Observed: 'Enter an integer: Odd'
Result: Fail

Test 3
Input: '0'
Expected: 'Even'
Obs

## Failure Analysis

One interesting failure appeared with Qwen2.5-Coder on Task 2.

The model generated valid Java code and the logic for finding the maximum value was reasonable. However, it did not follow the requested input method. Instead of reading the values from standard input, it tried to read them from `args[]`.

Because our predefined tests send the input through standard input, the program received no command-line arguments and crashed with an `ArrayIndexOutOfBoundsException`.

So the main issue was not the algorithm itself, but the fact that the model misunderstood one important instruction about how the input should be provided.